# Cursul 4. Curățarea și integrarea datelor
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

**Curățarea înseamnă verificarea datelor înainte de folosire.** În C3 am adus sursele la aceeași formă (preprocesarea); azi verificăm dacă valorile sunt corecte, complete și comparabile, apoi facem primul tabel de analiză. Pașii urmează etapele clasice ale curățării:

| Etapa | Întrebarea | În pandas | Secțiunea |
|---|---|---|---|
| Profilarea | Ce conține tabelul? | `info`, `value_counts`, `describe` | 2 |
| Tipurile | Numerele sunt numere, codurile sunt text? | `dtypes`, `pd.to_numeric` | 3 |
| Duplicatele | Apare aceeași cheie de două ori? | `duplicated`, `drop_duplicates` | 4 |
| Valorile imposibile | Există procente peste 100, valori negative, valori neobișnuite? | `describe`, `nsmallest`, un filtru | 5 |
| Lipsurile | Ce înseamnă „lipsă”, câte valori lipsesc și unde? | `isna`, `count` | 6 |
| Codurile | Aceeași țară are același cod în toate sursele? | `replace` | 7 |
| Primul tabel de analiză | Cum ajungem la un rând pe regiune? | `pivot`, `merge` | 8 |
| Validarea | Tabelul final respectă regulile lui? | o celulă de verificări | 9 |

Azi facem un singur tabel de analiză, cel al regiunilor. Perechile de regiuni și de țări (integrarea tuturor surselor), baza de date și agregarea vin în C5.

In [ ]:
import pandas as pd

## 1. Deschidem tabelele din C3
Ca în C3, le citim din depozitul cursului: sunt exact tabelele salvate de caietul 3, deci toți lucrăm pe aceleași date. Adresa folderului e aceeași, se schimbă doar numele fișierului.

In [ ]:
DATE_C3 = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/processed/"   # tabelele din C3

indicatori = pd.read_csv(DATE_C3 + "indicatori.csv")      # șapte indicatori, pe regiuni NUTS2
nume = pd.read_csv(DATE_C3 + "regiuni.csv")               # nomenclatorul: codul, țara, numele, nivelul
sci = pd.read_csv(DATE_C3 + "sci_regiuni.csv")            # SCI pe perechi de regiuni

### Ce conține fiecare tabel
Înainte de curățare, ne uităm la fiecare tabel: ce înseamnă un rând și ce coloane are.

| Tabelul | Un rând înseamnă | Coloanele |
|---|---|---|
| `indicatori` | o regiune NUTS 2 și un indicator | `regiune`, `valoare`, `indicator` |
| `nume` | un cod de regiune (de la țară până la NUTS 3) | `regiune`, `tara`, `denumire`, `nivel` |
| `sci` | o pereche de regiuni | `tara_1`, `tara_2`, `regiune_1`, `regiune_2`, `sci` |

`head()` arată primele 5 rânduri ale unui tabel:

In [ ]:
indicatori.head()

Tabelul este în **format lung**: valorile tuturor indicatorilor stau în aceeași coloană, `valoare`, iar coloana `indicator` spune ce măsoară fiecare. Aceeași regiune apare deci de mai multe ori, câte o dată pentru fiecare indicator. De exemplu, Nord-Vest (`RO11`):

In [ ]:
indicatori[indicatori["regiune"] == "RO11"]      # Nord-Vest: un rând pe indicator

Tabelul nu are o coloană pentru an, pentru că toate valorile sunt din același an, **2024**: în C3 am cerut fiecare indicator cu filtrul `"time": 2024`. Regiunile fără date pentru 2024 au valoarea lipsă, nu una dintr-un alt an.

Populația e în persoane, densitatea în locuitori pe km², PIB-ul pe locuitor în standarde ale puterii de cumpărare (PPS, un euro ajustat la nivelul prețurilor din fiecare țară), vârsta medie în ani, iar studiile superioare, șomajul și rețelele sociale în procente. Unitățile sunt diferite, deci valorile se compară doar în interiorul aceluiași indicator.

Nomenclatorul `nume` leagă fiecare cod de țară și de numele regiunii; `nivel` spune ce fel de unitate e.

In [ ]:
nume.head()

**NUTS** este împărțirea teritorială a Eurostat, pe niveluri încadrate unul în altul. Fiecare nivel adaugă un caracter la cod, deci lungimea codului arată nivelul (ca în C3). În România:

| `nivel` | Ce este | În România | Exemplu |
|---|---|---|---|
| 0 | țara | 1 | `RO`, România |
| 1 | macroregiunile (NUTS 1) | 4 | `RO1`, Macroregiunea Unu |
| 2 | regiunile de dezvoltare (NUTS 2) | 8 | `RO11`, Nord-Vest |
| 3 | județele, plus București (NUTS 3) | 42 | `RO113`, Cluj |

Regulamentul NUTS fixează pragurile de populație: 3-7 milioane de locuitori pentru NUTS 1, 0,8-3 milioane pentru NUTS 2, 150.000-800.000 pentru NUTS 3. De aceea o țară mică, precum Luxemburg, este o singură regiune pe toate nivelurile. Analiza noastră lucrează pe **NUTS 2**.

Codurile care încep cu `RO1` arată încadrarea: macroregiunea, cele două regiuni ale ei, apoi județele fiecăreia.

In [ ]:
nume[nume["regiune"].str.startswith("RO1")].sort_values("regiune")      # RO1, RO11 cu județele, RO12 cu județele

În `sci`, fiecare rând este o pereche de regiuni cu indicele lor de conectare socială: cu cât `sci` e mai mare, cu atât prieteniile pe Facebook între cele două regiuni sunt mai probabile. Indicele este relativ, deci are sens doar comparat cu alte perechi.

In [ ]:
sci.head()

**Observă:** primele două rânduri descriu aceeași pereche, `PT11` și `PL42`, în ordini diferite, cu aceeași valoare. Ne întoarcem la ele în secțiunea 4, la duplicate.

## 2. Profilarea: ce conține tabelul
Înainte de orice decizie, descriem tabelul, fără să-l modificăm. Primele întrebări: cât de mare e, ce coloane are, cum se termină și câte valori are fiecare coloană.

| Metoda | Ce arată |
|---|---|
| `shape` | (rânduri, coloane); fără paranteze, e un atribut |
| `columns` | numele coloanelor |
| `head()`, `tail()` | primele, respectiv ultimele 5 rânduri (sau câte cerem) |
| `info()` | tipul fiecărei coloane și câte valori are |
| `describe()` | minimul, media, cuartilele și maximul coloanelor numerice |
| `value_counts()` | de câte ori apare fiecare valoare dintr-o coloană |
| `nunique()`, `unique()` | câte valori distincte are o coloană, respectiv care sunt |

In [ ]:
print(indicatori.shape)                # (rânduri, coloane)
list(indicatori.columns)               # numele coloanelor

In [ ]:
indicatori.tail(3)                     # ultimele 3 rânduri: tabelul se termină cu rețelele sociale

`info()` adună într-un singur rezumat tipurile și numărul de valori din fiecare coloană:

In [ ]:
indicatori.info()

**Interpretează:** 2.281 de rânduri, dar coloana `valoare` are doar 1.893 de valori (*Non-Null Count*). Diferența, 388, sunt valori lipsă. `info()` arată dintr-o privire și tipul fiecărei coloane.

`value_counts()` numără de câte ori apare fiecare valoare dintr-o coloană. Pe `indicator`, arată câte regiuni are fiecare indicator.

In [ ]:
indicatori["indicator"].value_counts()

Cu `normalize=True`, `value_counts` dă proporții în loc de număr de rânduri. `nunique()` spune câte valori distincte are fiecare coloană, iar `unique()` le arată pe toate:

In [ ]:
print(indicatori.nunique())                  # câte valori distincte, pe coloană
indicatori["indicator"].unique()            # care sunt indicatorii

Indicatorii nu acoperă același număr de regiuni: de la 354 (vârsta medie) la 226 (rețelele sociale). E primul semn că lipsurile nu sunt distribuite uniform.

O verificare rapidă a codurilor: lungimea codului spune nivelul (C3). Toate ar trebui să aibă 4 caractere, adică NUTS2.

In [ ]:
indicatori["regiune"].str.len().value_counts()     # lungimea codurilor, numărată

`describe()` funcționează și pe coloanele text, cu `include="object"`: arată câte valori sunt, câte sunt distincte și care apare cel mai des.

In [ ]:
nume.describe(include="object")

**Interpretează:** codurile sunt toate diferite (1.971 de valori, 1.971 distincte), dar numele nu: „Kýpros” (Cipru) apare de 4 ori, o dată pe fiecare nivel, de la țară până la NUTS 3. De aceea tabelele se unesc după **coduri**, niciodată după nume.

**Exercițiu.** Nomenclatorul `nume` conține toate nivelurile. Câte coduri are fiecare nivel?

In [ ]:
# Câte coduri are fiecare nivel: value_counts pe coloana nivel, apoi sort_index()

### O funcție proprie, cu `apply`
Nivelul se poate citi și din cod: `RO` are 2 caractere (nivelul 0), `RO1` are 3 (nivelul 1), `RO11` are 4, `RO113` are 5. Deci nivelul = lungimea codului - 2. Scriem regula ca **funcție**, apoi `apply` o aplică pe fiecare valoare din coloană:

In [ ]:
def nivel_din_cod(cod):
    return len(cod) - 2                 # RO -> 0, RO1 -> 1, RO11 -> 2, RO113 -> 3

nivel_din_cod("RO113")                  # încercăm funcția pe un singur cod

In [ ]:
nume["nivel_calculat"] = nume["regiune"].apply(nivel_din_cod)    # funcția, pe fiecare cod
(nume["nivel_calculat"] != nume["nivel"]).sum()                   # câte nu se potrivesc cu coloana nivel

**Interpretează:** 0. Coloana `nivel` e consecventă cu codurile, deci ne putem baza pe ea. `apply` e util când regula e prea complicată pentru o singură metodă: o funcție poate avea `if`, mai mulți pași, orice. Când există o metodă directă, ea e mai simplă și mai rapidă; aici, `nume["regiune"].str.len() - 2` dă același rezultat.

## 3. Tipurile
**Anticipează:** ce tip are coloana `valoare`? Dar `regiune`?

In [ ]:
indicatori.dtypes

`valoare` este `float64`, număr cu zecimale; o valoare lipsă apare ca `NaN`. `regiune` și `indicator` sunt `object`, adică text.

Dacă printre numere apare un text (Eurostat scrie uneori `:` în loc de o valoare), toată coloana devine `object` și calculele nu mai merg. `pd.to_numeric` transformă textul în numere, iar cu `errors="coerce"` ce nu e număr devine valoare lipsă:

In [ ]:
exemplu = pd.Series(["12.5", ":", "7"])           # o coloană citită ca text
pd.to_numeric(exemplu, errors="coerce")            # ":" devine NaN, restul devin numere

Când știm sigur că toate valorile se pot converti, `astype` schimbă tipul direct; dacă o valoare nu se poate converti, dă eroare, deci e și o verificare:

In [ ]:
pd.Series(["2022", "2023", "2024"]).astype(int)    # anii, din text în numere întregi

## 4. Duplicatele
Un rând din `indicatori` e identificat de **regiune + indicator**: aceasta e cheia tabelului. Nicio cheie nu trebuie să apară de două ori. `duplicated` marchează rândurile care repetă o cheie deja întâlnită.

In [ ]:
# Câte rânduri repetă cheia regiune + indicator: duplicated cu lista celor două coloane, apoi .sum()

0: fiecare cheie apare o singură dată.

În `sci`, un rând este o pereche de regiuni, deci cheia are tot două coloane: `regiune_1` + `regiune_2`. Câte rânduri are tabelul și câte chei se repetă?

In [ ]:
len(sci)                                                # câte rânduri

In [ ]:
sci.duplicated(["regiune_1", "regiune_2"]).sum()      # câte chei repetate

Niciuna. Asta nu înseamnă că fiecare legătură apare o singură dată. Ne uităm la o pereche concretă, Nord-Vest (`RO11`) și Lombardia (`ITC4`).

**Pasul 1.** Rândul în care Nord-Vest e prima regiune și Lombardia a doua. Folosim două condiții, legate cu `&` („și”), fiecare între paranteze:

In [ ]:
sci[(sci["regiune_1"] == "RO11") & (sci["regiune_2"] == "ITC4")]

**Pasul 2.** Aceeași pereche, în ordine inversă:

In [ ]:
sci[(sci["regiune_1"] == "ITC4") & (sci["regiune_2"] == "RO11")]

**Interpretează:** două rânduri, cu aceeași valoare, 9.760. Cheile sunt diferite (`RO11`-`ITC4` și `ITC4`-`RO11`), deci `duplicated` nu le vede, dar descriu aceeași legătură. Este o dublură de **conținut**, nu de cheie.

### Datele diadice: de ce apar dublurile
Până acum, un rând era o regiune. În `sci`, un rând este o **pereche** de regiuni (o *diadă*): valoarea nu descrie un loc, ci legătura dintre două locuri.

Meta publică SCI ca pe un tabel cu toate combinațiile: fiecare regiune cu fiecare regiune, inclusiv cu ea însăși. Pentru trei regiuni, asta înseamnă 3 × 3 = 9 rânduri:

In [ ]:
trei = ["RO11", "RO12", "ITC4"]                                     # Nord-Vest, Centru, Lombardia
noua = sci[sci["regiune_1"].isin(trei) & sci["regiune_2"].isin(trei)]   # toate combinațiile dintre ele
noua

Așezate ca tabel, cele 9 valori arată așa:

| | `ITC4` | `RO11` | `RO12` |
|---|---|---|---|
| **`ITC4`** | *929.834* | 9.760 | 4.268 |
| **`RO11`** | 9.760 | *5.032.471* | 308.881 |
| **`RO12`** | 4.268 | 308.881 | *5.051.839* |

Fiecare valoare din afara diagonalei apare **de două ori**, o dată deasupra și o dată dedesubt:

| Perechea | În tabel apare la | SCI |
|---|---|---|
| Nord-Vest și Centru | rândul `RO11`, coloana `RO12` **și** rândul `RO12`, coloana `RO11` | 308.881 |
| Nord-Vest și Lombardia | rândul `RO11`, coloana `ITC4` **și** rândul `ITC4`, coloana `RO11` | 9.760 |
| Centru și Lombardia | rândul `RO12`, coloana `ITC4` **și** rândul `ITC4`, coloana `RO12` | 4.268 |

SCI măsoară prieteniile *dintre* două regiuni, iar o prietenie nu are direcție. Deci din 9 rânduri: **3 perechi** distincte (fiecare de două ori, adică 6 rânduri) și **3 rânduri pe diagonală** (cu italic în tabel), fiecare regiune cu ea însăși. Diagonala descrie prieteniile din interiorul regiunii, nu o legătură între două locuri, deci nu ne interesează.

Pentru toate cele 298 de regiuni: 298 × 298 = 88.804 de rânduri, dintre care 298 pe diagonală și (88.804 - 298) / 2 = **44.253 de perechi** distincte.

### De ce le scoatem și cum
Dacă am păstra ambele ordini, fiecare legătură ar intra de două ori în orice calcul: media SCI ar fi calculată pe date dublate, iar în model (C7) am avea de două ori mai multe observații decât în realitate.

`drop_duplicates` nu ne ajută, pentru că cele două rânduri ale unei perechi nu sunt identice: diferă ordinea regiunilor. Folosim altă regulă: păstrăm doar rândul în care primul cod e **mai mic alfabetic** decât al doilea. Pentru Nord-Vest și Lombardia, `"ITC4" < "RO11"`, deci rămâne rândul `ITC4`-`RO11`. Pe diagonală, un cod nu e mai mic decât el însuși, deci dispare și diagonala:

In [ ]:
noua[noua["regiune_1"] < noua["regiune_2"]]      # din 9 rânduri rămân 3: câte unul pe pereche

În C5 aplicăm aceeași regulă pe tot tabelul `sci`.

### Cum se va lega totul
Unitatea de analiză a proiectului este perechea. În C5, fiecărei perechi îi lipim indicatorii **ambelor** regiuni (populația regiunii 1, populația regiunii 2 etc.); în C7 construim din ei variabile ale perechii (de exemplu, diferența de vârstă medie) și întrebăm ce explică SCI.

Migrația e diferită: are direcție (românii din Italia nu sunt același lucru cu italienii din România). De aceea, la perechile de țări (tot în C5), păstrăm tot un rând pe pereche, dar cu migrația în **două coloane**, câte una pentru fiecare sens.

## 5. Valorile imposibile sau neobișnuite
Luăm un singur indicator, populația, și ne uităm la rezumatul lui: minimul, cuartilele, maximul.

In [ ]:
populatie = indicatori[indicatori["indicator"] == "populatie"]     # doar rândurile cu populația
populatie["valoare"].describe().round(1)

**Interpretează:** populația minimă este 0. Care regiune e?

In [ ]:
populatie.nsmallest(3, "valoare")          # cele mai mici trei valori

`NO0B` este Svalbard, arhipelagul norvegian: Eurostat raportează chiar 0. Nu e o eroare. **Valorile neobișnuite nu se șterg automat**: întâi se verifică în sursă.

Verificăm și celălalt capăt, cu `sort_values`, care ordonează tabelul după o coloană (`ascending=False` înseamnă descrescător):

In [ ]:
populatie.sort_values("valoare", ascending=False).head(3)     # cele mai mari trei valori

Istanbul (`TR10`), Île-de-France (`FR10`) și Lombardia (`ITC4`): valori mari, dar plauzibile.

**Exercițiu.** Studiile superioare, șomajul și rețelele sociale sunt procente, deci între 0 și 100. Câte valori sunt peste 100?

In [ ]:
# 1. Rândurile cu indicatorii în procente: studii_superioare, somaj, retele_sociale (cu isin)
# 2. Câte valori sunt peste 100? (o condiție, apoi .sum())

## 6. Lipsurile: ce înseamnă „lipsă”
Pentru pandas, o valoare lipsă e `NaN`. La citire, pandas transformă automat în `NaN` câteva texte: celula goală, dar și `NA`, `N/A`, `null`. La noi, `NA` este codul Namibiei. Comparăm cele două citiri ale migrației:

In [ ]:
gresit = pd.read_csv(DATE_C3 + "migratie.csv")                                           # citirea implicită
migratie = pd.read_csv(DATE_C3 + "migratie.csv", keep_default_na=False, na_values=[""])  # doar celula goală e lipsă

print("cetățenii lipsă, citirea implicită:", gresit["cetatenie"].isna().sum())
print("cetățenii lipsă, citirea corectă:  ", migratie["cetatenie"].isna().sum())

44 de rânduri cu cetățenii namibieni ar fi devenit „lipsă”, fără niciun mesaj. De aceea codurile de țară se citesc mereu cu `keep_default_na=False`.

`value_counts()` ascunde implicit valorile lipsă. Cu `dropna=False`, le numără și pe ele:

In [ ]:
zboruri = pd.read_csv(DATE_C3 + "zboruri.csv", keep_default_na=False, na_values=[""])
# Câte perechi au și câte nu au pasageri: isna() pe coloana pasageri, apoi value_counts(dropna=False)

## 7. Codurile
O capcană care nu dă nicio eroare: Eurostat scrie Grecia `EL` și Regatul Unit `UK`, iar SCI le scrie `GR` și `GB`. Dacă nu le aliniem, aceste țări se pierd la lipire, fără niciun mesaj. `replace` înlocuiește valorile după un dicționar.

In [ ]:
coduri = {"EL": "GR", "UK": "GB"}                    # codul Eurostat -> codul SCI

migratie = migratie.replace({"tara_rezidenta": coduri, "cetatenie": coduri})    # în ambele coloane
zboruri = zboruri.replace({"tara_1": coduri, "tara_2": coduri})

migratie["tara_rezidenta"].isin(["EL", "UK"]).sum()     # verificare: nu mai rămâne niciun EL sau UK

Codurile noastre vin de la API-uri și sunt curate. În datele introduse de mână, aceeași categorie apare des scrisă în mai multe feluri: cu spații, cu litere mici. Metodele `.str` le aduc la aceeași formă:

In [ ]:
scrise = pd.Series([" RO", "ro", "Ro "])          # aceeași țară, scrisă în trei feluri
print(scrise.nunique(), "valori distincte")
scrise.str.strip().str.upper()                    # fără spații, cu majuscule: "RO" de trei ori

## 8. Tabelul regiunilor: din lung în lat
> **Secțiunea 8 o rulăm împreună.** Nu are exerciții: rulați celulele pe rând și urmăriți, după fiecare `merge`, numărul de rânduri.

Lucrăm doar cu regiunile din SCI, aria proiectului. Întâi lista lor: în coloana `regiune_1`, fiecare regiune apare de 298 de ori, o dată cu fiecare partener; `drop_duplicates` păstrează fiecare valoare o singură dată. (Aici lucrează pe **o singură coloană**, unde valorile repetate chiar sunt identice.)

In [ ]:
regiuni_sci = sci["regiune_1"].drop_duplicates()                    # fiecare regiune, o singură dată
in_sci = indicatori[indicatori["regiune"].isin(regiuni_sci)]        # doar indicatorii regiunilor din SCI
len(regiuni_sci)

Trecem din formatul **lung** (un rând pe regiune și indicator) în formatul **lat** (un rând pe regiune, o coloană pe indicator). `pivot` face asta. Nu calculează nimic: doar mută fiecare valoare în căsuța ei.

In [ ]:
valori = in_sci.pivot(index="regiune", columns="indicator", values="valoare")   # rânduri, coloane, celule
valori.columns.name = None                                                      # fără eticheta "indicator" deasupra coloanelor
valori = valori.reset_index()                                                   # regiunea, din nou coloană
valori.head()

**Observă:** apar multe `NaN`. Nu e o greșeală a lui `pivot`: unde în tabelul lung nu exista o valoare pentru o regiune și un indicator, căsuța din tabelul lat rămâne goală. Albania (`AL01`-`AL03`), de exemplu, are doar densitatea; Eurostat nu publică pentru ea ceilalți indicatori la nivel NUTS 2.

Ce facem cu ele? Întâi le numărăm și vedem unde sunt; abia apoi decidem. Facem asta imediat după ce construim tabelul regiunilor.

Acum construim **tabelul regiunilor**: un rând pe regiune, cu codul, țara, numele și cei șapte indicatori. Piesele sunt în trei tabele diferite:

| Tabelul | Ce aduce | Câte rânduri |
|---|---|---|
| `regiuni_sci` | lista regiunilor proiectului (cele din SCI) | 298 |
| `nume` | țara și numele fiecărui cod | 1.971 (toate nivelurile, toată Europa) |
| `valori` | cei șapte indicatori, pe regiune | câte regiuni au date |

Le lipim pe rând, cu `merge`, după coloana comună `regiune`. Pornim de la listă, pentru că ea spune ce regiuni vrem: `how="left"` păstrează toate rândurile din stânga (cele 298) și aduce din dreapta doar ce se potrivește. Ce nu se găsește în dreapta rămâne `NaN`, nu dispare.

**Pasul 1.** Lista, transformată în tabel cu o singură coloană:

In [ ]:
regiuni = pd.DataFrame({"regiune": regiuni_sci})       # 298 de rânduri, o coloană
print(len(regiuni), "rânduri")
regiuni.head(3)

**Pasul 2.** Lipim țara și numele. Din `nume` luăm doar cele trei coloane de care avem nevoie; din cele 1.971 de coduri, `merge` le aduce doar pe cele 298 din listă.

In [ ]:
regiuni = regiuni.merge(nume[["regiune", "tara", "denumire"]], on="regiune", how="left")
print(len(regiuni), "rânduri")                         # tot 298
regiuni.head(3)

**Pasul 3.** Lipim indicatorii. O regiune fără date pentru un indicator primește `NaN` în coloana lui.

In [ ]:
regiuni = regiuni.merge(valori, on="regiune", how="left")
print(len(regiuni_sci), "->", len(regiuni), "regiuni")     # regula integrării: același număr
regiuni.head(3)

Același număr: fiecare regiune a primit cel mult un rând. Dacă numărul ar crește, o regiune ar fi apărut de două ori în tabelul lipit; dacă ar scădea, am fi pierdut regiuni.

Câte valori lipsesc, pe coloană, și ce procent reprezintă:

In [ ]:
lipsuri = pd.DataFrame({"lipsa": regiuni.isna().sum(),                       # câte
                        "procent": (regiuni.isna().mean() * 100).round(1)})  # ce procent din regiuni
lipsuri

`isna()` dă `True` unde lipsește o valoare; `sum()` numără valorile `True`, iar `mean()` dă proporția lor. Rețelele sociale lipsesc pentru 42% dintre regiuni, restul indicatorilor pentru sub 8%.

**Ce s-ar întâmpla dacă am elimina lipsurile?** `dropna()` scoate fiecare rând care are cel puțin o valoare lipsă:

In [ ]:
complete = regiuni.dropna()                                   # doar regiunile fără nicio lipsă
print(len(complete), "din", len(regiuni), "regiuni")
print(complete["tara"].nunique(), "din", regiuni["tara"].nunique(), "țări")

Ar rămâne 161 de regiuni din 298 și 24 de țări din 37: am pierde aproape jumătate din date și țări întregi.

**Și dacă le-am completa?** `fillna(valoare)` pune o valoare în locul fiecărei lipse, de exemplu mediana. Comparăm abaterea standard a PIB-ului înainte și după:

In [ ]:
pib = regiuni["pib_locuitor"]
completat = pib.fillna(pib.median())                          # fiecare lipsă devine mediana
print("abaterea standard, cu lipsurile:  ", round(pib.std()))
print("abaterea standard, după completare:", round(completat.std()))

Completarea face datele să pară mai omogene decât sunt (abaterea standard scade de la 14.503 la 13.968), pentru că 22 de regiuni primesc exact aceeași valoare, inventată.

Dacă am păstra doar regiunile complete, am pierde multe regiuni, iar pierderea nu e întâmplătoare: lipsurile sunt concentrate în câteva țări. PIB-ul lipsește în Elveția, Norvegia, Albania, Bosnia și Herțegovina, Kosovo și Islanda; rețelele sociale lipsesc mai ales în Germania, Franța și Turcia, care nu le raportează pe regiuni NUTS 2.

**Decizia noastră: păstrăm toate regiunile, fără să completăm valorile lipsă.** Lipsurile se tratează abia la analiză, și doar pentru coloanele folosite: `dropna(subset=[...])` scoate rândurile fără valori **doar în coloanele din listă**. O analiză a PIB-ului și a șomajului pierde astfel 26 de regiuni, nu 137:

In [ ]:
pentru_analiza = regiuni.dropna(subset=["pib_locuitor", "somaj"])    # doar coloanele de care are nevoie analiza
print(len(regiuni), "->", len(pentru_analiza), "regiuni")

Tabelul salvat rămâne întreg; fiecare analiză din C6 și C7 își alege singură rândurile. Decizia se notează în jurnal (secțiunea 10).

## 9. Validarea finală
Înainte de salvare, verificăm regulile tabelului regiunilor: câte rânduri are, dacă cheia lui e unică și dacă fiecare regiune are o țară.

In [ ]:
print(len(regiuni), "regiuni")                                   # trebuie 298
print(regiuni.duplicated("regiune").sum(), "chei repetate")       # trebuie 0
print(regiuni["tara"].isna().sum(), "regiuni fără țară")          # trebuie 0

Rezultatul așteptat: 298 de regiuni, 0 chei repetate, 0 regiuni fără țară. Dacă o cifră diferă, un pas de mai sus trebuie refăcut înainte de salvare.

## 10. Salvăm și scriem deciziile
Tabelul regiunilor și cele două tabele cu codurile aliniate, în `data/processed`. Din ele pornim în C5.

In [ ]:
regiuni.to_csv("data/processed/regiuni_nuts2.csv", index=False)        # un rând pe regiune
migratie.to_csv("data/processed/migratie_curata.csv", index=False)     # codurile GR și GB, Namibia păstrată
zboruri.to_csv("data/processed/zboruri_curate.csv", index=False)       # codurile GR și GB

### Jurnalul deciziilor
Orice curățare înseamnă decizii. Le scriem, cu motivul lor, ca oricine să le poată verifica (în proiect, în `README.md`).

| Tema | Decizia noastră | De ce |
|---|---|---|
| Valori neobișnuite (populația 0 la Svalbard) | păstrate | sunt raportate așa de sursă |
| Codul `NA` (Namibia) | citit ca text | altfel 44 de rânduri devin lipsă |
| Regiuni fără date | păstrate, fără completare | lipsurile nu sunt întâmplătoare |
| Coduri EL/UK | aliniate la GR/GB | altfel pierdem țări la lipire |
| Perechi duble A-B / B-A | de rezolvat în C5 | aceeași valoare, de două ori |

### Exercițiul echipei (dacă rămâne timp)
1. Din ce țări sunt regiunile fără PIB?

In [ ]:
# 1. Regiunile în care pib_locuitor lipsește (isna)
# 2. Câte sunt, pe țară (value_counts pe coloana tara)

2. Gândiți-vă la datele proiectului vostru: care dintre problemele de azi (tipuri, duplicate, valori imposibile, lipsuri, coduri) ar putea apărea? Notați una sau două, cu verificarea potrivită. Curățarea lor se face ulterior.

## Ce ducem mai departe
În `data/processed` avem tabelul regiunilor și migrația și zborurile cu codurile aliniate. În C5 unim toate sursele în perechi de regiuni și de țări, le încărcăm într-o bază de date și le agregăm, în SQL și în Python.

**Poți explica fără cod?** Ce arată `info()` și ce arată `value_counts()`? Prin ce diferă o cheie repetată de o dublură de conținut? De ce `NA` nu e întotdeauna o valoare lipsă? Ce face `how="left"`? De ce numărăm rândurile după fiecare `merge`? Ce face `apply`? Ce verifică celula de validare finală?

### Surse
- [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas: Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)
- [pandas: Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)
- [Eurostat, Glossary: Country codes](https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Glossary:Country_codes)